# 02 - Fetch Candidate Committees and Totals

This notebook loads the Senate candidate universe and fetches each candidate's committee links and finance totals from OpenFEC.
We save both the committee mapping table and the candidate finance totals table.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

from midterm_money.config import settings
from midterm_money.fec_client import FECClient


In [ ]:
candidates_path = ROOT / "data" / "processed" / "senate_candidates_2026_all.csv"
df_candidates = pd.read_csv(candidates_path, dtype=str)
print("Loaded candidates table shape:", df_candidates.shape)
df_candidates.head()

In [ ]:
client = FECClient()
committee_rows = []
totals_rows = []
missing_committees = []
missing_totals = []

for _, row in df_candidates.iterrows():
    candidate_id = row.get("fec_candidate_id")
    name = row.get("name")
    if not candidate_id:
        continue

    committee_data = None
    totals_data = None

    try:
        committee_data = client.get(f"/candidate/{candidate_id}/committees/", params={"per_page": 100})
    except Exception as exc:
        missing_committees.append((candidate_id, str(exc)))

    try:
        totals_data = client.get(f"/candidate/{candidate_id}/totals/")
    except Exception as exc:
        try:
            totals_data = client.get(f"/candidates/{candidate_id}/totals/")
        except Exception as exc2:
            missing_totals.append((candidate_id, str(exc) + " | " + str(exc2)))

    committee_results = committee_data.get("results", []) if committee_data else []
    for committee in committee_results:
        committee_rows.append({
            "fec_candidate_id": candidate_id,
            "candidate_name": name,
            "committee_id": committee.get("committee_id"),
            "committee_name": committee.get("name"),
            "designation": committee.get("designation"),
            "designation_full": committee.get("designation_full"),
            "committee_type": committee.get("committee_type"),
            "committee_type_full": committee.get("committee_type_full"),
            "is_principal": (committee.get("designation") == "P" or (committee.get("designation_full") or "").lower().find("principal") >= 0),
        })

    totals_results = totals_data.get("results", []) if totals_data else []
    if totals_results:
        for totals in totals_results:
            totals_rows.append({
                "fec_candidate_id": candidate_id,
                "candidate_name": name,
                "state": row.get("state"),
                "party": row.get("party"),
                "committee_id": totals.get("committee_id"),
                "total_receipts": totals.get("total_receipts"),
                "total_disbursements": totals.get("total_disbursements"),
                "cash_on_hand_end_period": totals.get("cash_on_hand_end_period"),
                "cash_on_hand": totals.get("cash_on_hand"),
                "debts_owed_by_committee": totals.get("debts_owed_by_committee"),
                "coverage_end_date": totals.get("coverage_end_date"),
                "cycle": totals.get("cycle"),
                "source_endpoint": totals_data.get("results") and "candidate_totals" or "unknown",
            })
    else:
        missing_totals.append((candidate_id, "No totals returned"))


In [ ]:
df_committees = pd.DataFrame(committee_rows)
df_totals = pd.DataFrame(totals_rows)
print("Committee rows:", len(df_committees))
print("Totals rows:", len(df_totals))
df_committees.head()

In [ ]:
processed_committees = ROOT / "data" / "processed" / "senate_candidate_committees_2026.csv"
processed_totals = ROOT / "data" / "processed" / "senate_candidate_finance_totals_2026.csv"
outputs_committees = ROOT / "outputs" / "senate_candidate_committees_2026.csv"
outputs_totals = ROOT / "outputs" / "senate_candidate_finance_totals_2026.csv"
processed_committees.parent.mkdir(parents=True, exist_ok=True)
processed_totals.parent.mkdir(parents=True, exist_ok=True)
outputs_committees.parent.mkdir(parents=True, exist_ok=True)
outputs_totals.parent.mkdir(parents=True, exist_ok=True)
df_committees.to_csv(processed_committees, index=False)
df_committees.to_csv(outputs_committees, index=False)
df_totals.to_csv(processed_totals, index=False)
df_totals.to_csv(outputs_totals, index=False)
print("Saved committees to:", processed_committees)
print("Saved totals to:", processed_totals)

In [ ]:
print("Missing committees count:", len(missing_committees))
print("Missing totals count:", len(missing_totals))
print("Top 20 candidates by total receipts:")
if not df_totals.empty:
    display(df_totals.sort_values("total_receipts", ascending=False).head(20))

print("States with no DEM or REP candidate in totals data:")
counts = df_totals.assign(party_normalized=df_totals["party"].str.upper().map({"D": "DEM", "DEM": "DEM", "DEMOCRAT": "DEM", "DEMOCRATIC": "DEM", "R": "REP", "REP": "REP", "REPUBLICAN": "REP"}).fillna("OTHER")).groupby(["state", "party_normalized"]).size().unstack(fill_value=0)
display(counts[counts.get("DEM", 0) == 0 | counts.get("REP", 0) == 0].head(20))